# PA4 Task 2 — mechanism diagnostic (supplementary, ~45 min on 2 × T4)

Runs **after** the main Task 2 run and changes no main-run condition. It explains three Part B findings and one Part C
finding with causal tests (`scripts/diagnose_task2.py`, `src/diag_b.py`, `src/diag_c.py`):

| GPU 0 — Part B (MQAR) | GPU 1 — Part C (KV cache) |
|---|---|
| failure taxonomy and slot accuracy, training plateaus, attention circuit scores, head ablation, delta-rule write-gate interventions, ELU+1 filler control, pair-block relocation, 17 retraining runs (supervision masking, near-data controls, 5 new linear seeds) | 11 interventions on token-wise INT4 keys (outlier channels, channel centring, sink tokens, INT8, layer localisation, K-only / V-only) with exact match, NLL, KL and per-layer attention distortion on all 180 prompts |

Outputs: `results/diagnostics/` (JSON + retraining runs), tables D1–D9 and figures FD1–FD4, zipped as `task2_diagnostics.zip`.

**Before running**
1. The main run's `PA4/Task 2/results/` folder must be committed and pushed (the notebook clones the repo).
2. Upload `task2_mqar_checkpoints.zip` (from the main run) as a Kaggle dataset and add it to this notebook
   (**Add Input → Your Datasets**). The notebook finds the 42 `.pt` files anywhere under `/kaggle/input`.
3. Settings: **Accelerator = GPU T4 × 2**, **Internet = On**.

In [ ]:
REPO_URL = "https://github.com/wasaymj/ATML.git"
BRANCH   = "main"

import os, subprocess, sys, shutil, json, glob, time, zipfile
WORK = "/kaggle/working"
REPO = f"{WORK}/ATML"
TASK = f"{REPO}/PA4/Task 2"
DATA = f"{REPO}/PA4/atml-pa4-datasets"
LOGS = f"{WORK}/logs"
CKPT, CACHE = f"{WORK}/mqar_checkpoints", f"{WORK}/mqar_cache"
NL = chr(10)
os.makedirs(LOGS, exist_ok=True)
print(subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, REPO], check=True)
print(subprocess.run(["git", "-C", REPO, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
need = ["results/part_b/test_query_table.npz", "results/part_b/runs/delta_K16_mixed_s6304/summary.json",
        "results/part_c/per_prompt.jsonl.gz"]
missing = [p for p in need if not os.path.exists(f"{TASK}/{p}")]
if not os.path.exists(f"{DATA}/data/local_source_revisions.json"):
    missing.append("atml-pa4-datasets/data/local_source_revisions.json")
assert not missing, f"commit and push the main Task 2 results first: {missing}"

In [ ]:
# checkpoints: a Kaggle dataset made from task2_mqar_checkpoints.zip (auto-extracted or still zipped), or a repo copy
os.makedirs(CKPT, exist_ok=True)
found = glob.glob("/kaggle/input/**/dense_K16_mixed_s6304.pt", recursive=True)
if found:
    for f in glob.glob(os.path.join(os.path.dirname(found[0]), "*.pt")):
        shutil.copy(f, CKPT)
else:
    zips = glob.glob("/kaggle/input/**/task2_mqar_checkpoints*.zip", recursive=True)
    if zips:
        zipfile.ZipFile(zips[0]).extractall(CKPT)
    elif os.path.isdir(f"{TASK}/checkpoints"):
        for f in glob.glob(f"{TASK}/checkpoints/*.pt"):
            shutil.copy(f, CKPT)
n_ckpt = len(glob.glob(f"{CKPT}/*.pt"))
print(n_ckpt, "checkpoints in", CKPT)
assert n_ckpt == 42, "add task2_mqar_checkpoints.zip as a Kaggle dataset (Add Input) and rerun this cell"

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{TASK}/requirements.txt"], check=True)
rev = json.load(open(f"{DATA}/data/local_source_revisions.json"))["revisions"]["model_target"]
from huggingface_hub import snapshot_download
print("Qwen/Qwen2.5-1.5B-Instruct @", rev, "->", snapshot_download("Qwen/Qwen2.5-1.5B-Instruct", revision=rev))

In [ ]:
ENV = dict(os.environ, PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True", TOKENIZERS_PARALLELISM="false",
           HF_HUB_DISABLE_PROGRESS_BARS="1")

def launch(name, gpu, args):
    script = f"{LOGS}/{name}.sh"
    lines = [f'cd "{TASK}"', f'{sys.executable} scripts/diagnose_task2.py {args} || echo "STEP FAILED"']
    open(script, "w").write(NL.join(lines) + NL)
    print(f"[{name}] GPU {gpu}: {args}", flush=True)
    return subprocess.Popen(["bash", script], stdout=open(f"{LOGS}/{name}.log", "a"), stderr=subprocess.STDOUT,
                            env=dict(ENV, CUDA_VISIBLE_DEVICES=str(gpu)))

def wait(procs, every_min=5):
    while any(p.poll() is None for p in procs.values()):
        time.sleep(every_min * 60)
        for name in procs:
            tail = open(f"{LOGS}/{name}.log", errors="replace").read().splitlines()[-3:]
            print(f"--- {name} ({time.strftime('%H:%M')}):", *tail, sep=NL + "   ", flush=True)
    print({n: p.returncode for n, p in procs.items()})

B_ARGS = f'--part b --ckpt_dir "{CKPT}" --cache_dir "{CACHE}"'
C_ARGS = "--part c"
import torch
if torch.cuda.device_count() >= 2:
    wait({"diag_b": launch("diag_b", 0, B_ARGS), "diag_c": launch("diag_c", 1, C_ARGS)})
else:
    wait({"diag_b": launch("diag_b", 0, B_ARGS)})
    wait({"diag_c": launch("diag_c", 0, C_ARGS)})
for f in glob.glob(f"{LOGS}/*.log"):
    print(f, "<-- contains a failed step" if "STEP FAILED" in open(f, errors="replace").read() else "ok")

In [ ]:
rc = subprocess.run([sys.executable, "scripts/compile_task2_report.py"], cwd=TASK, env=ENV).returncode
print("report exit code", rc)
# package the diagnostic outputs (plus the regenerated tables/figures) for download
shutil.copytree(LOGS, f"{TASK}/results/diagnostics/logs", dirs_exist_ok=True)
out = f"{WORK}/task2_diagnostics"
shutil.rmtree(out, ignore_errors=True)
for sub in ["diagnostics", "tables", "figures"]:
    if os.path.isdir(f"{TASK}/results/{sub}"):
        shutil.copytree(f"{TASK}/results/{sub}", f"{out}/results/{sub}")
shutil.make_archive(f"{WORK}/task2_diagnostics", "zip", out, "results")
print(sorted(os.listdir(WORK)))

In [ ]:
# inline preview (display only; the zip above is already written)
from IPython.display import Markdown, Image, display
for t in sorted(glob.glob(f"{TASK}/results/tables/D*.md")):
    display(Markdown(f"### {os.path.basename(t)}" + NL + open(t, encoding="utf-8").read()))
for fig in sorted(glob.glob(f"{TASK}/results/figures/FD*.png")):
    display(Image(fig, width=900))